# RDT-1B Feature-level BlurIG
**What this does:** Runs ManiSkill simulation to get a real robot camera frame, then applies Blur Integrated Gradients in SigLIP patch-embedding space to show which parts of the image most influenced RDT's predicted gripper action.

**Before running:** Set `Runtime > Change runtime type > A100 GPU`

**Runtime:** ~5-10 min on A100 (first run downloads ~2GB of models)

In [ ]:
import os, sys, subprocess

# Verify mani_skill is intact — submodules can silently disappear between sessions.
# --no-deps reinstalls only the mani-skill package (not all dependencies).
# --no-cache-dir forces a fresh download instead of reusing a possibly-corrupted
# local pip cache (which is what made earlier reinstall attempts not actually fix it).
try:
    import mani_skill.utils
except (ImportError, ModuleNotFoundError):
    print('mani_skill broken — reinstalling (package only)...')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--force-reinstall',
         '--no-deps', '--no-cache-dir', 'mani-skill'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('Reinstall FAILED — pip output:')
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        raise RuntimeError('mani-skill reinstall failed — see pip output above')
    print('Reinstalled OK. Re-run this cell once more to continue.')
    os.kill(os.getpid(), 9)

if os.path.exists('/content/.deps_installed'):
    print('Already installed — skip to Cell 2.')
else:
    os.system('git clone https://github.com/thu-ml/RoboticsDiffusionTransformer')
    os.system('git clone https://github.com/rdgbrandon/rdt-igtesting')
    os.system('apt-get install -qq libvulkan1 libegl1-mesa-dev libgles2-mesa-dev libosmesa6-dev')
    os.system('pip install -q einops timm pyyaml mani-skill accelerate')
    os.system('pip install -q "numpy<2"')
    os.system('pip install -q "protobuf>=5.28.0"')
    open('/content/.deps_installed', 'w').close()
    print('Install done. Restarting runtime...')
    os.kill(os.getpid(), 9)

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

os.chdir("/content/rdt-igtesting")
os.system("git pull -q")

TASKS = ["PickCube-v1", "StackCube-v1", "PushCube-v1", "PegInsertionSide-v1", "PlugCharger-v1"]
for task in TASKS:
    shutil.copy(
        hf_hub_download("robotics-diffusion-transformer/maniskill-model",
                        "lang_embeds/text_embed_" + task + ".pt"),
        "lang_embed_" + task + ".pt"
    )
    print("Downloaded lang embed:", task)

shutil.copy("lang_embed_PickCube-v1.pt", "lang_embed.pt")
print("Ready.")

In [ ]:
# @title Rollout settings
n_successes = 1  # @param {type:"integer"}
task = "PickCube-v1"  # @param ["PickCube-v1", "StackCube-v1", "PushCube-v1", "PegInsertionSide-v1", "PlugCharger-v1"]

from run_rollout_cell import run
run(task=task, n=n_successes)

In [ ]:
import os
os.environ["SKIP_IG"] = "1"
os.environ["MANISKILL_TASK"] = task
os.environ["LANG_EMBED"] = "lang_embed_" + task + ".pt"
%run -i rdt_blurig.py
%run -i run_ig.py